Load the dataset with PySpark

In [ ]:
from pyspark.sql import SparkSession
from pathlib import Path
from pyspark.sql import functions as F
import country_converter as coco
from itertools import chain

CLEANED_DATA_DIR = Path("../data/cleaned")
PROCESSED_DATA_DIR = Path("../data")

spark = SparkSession.builder.appName("profiling").config("spark.sql.ansi.enabled", "false").config("spark.driver.memory", "16g").getOrCreate()
df = spark.read.parquet(f"{PROCESSED_DATA_DIR}/4_eea_co2_emissions_from_passenger_cars-001.parquet")

last_present = df.count()
print(f"{last_present} rows present")

Remap column names to more easily understandable ones

In [ ]:
column_mapping = {
    "Country": "geo",
    "VFN": "vehicle_family_id_number",
    "Mh": "manufacturer_name_eu_standard_denomination",
    "T": "type",
    "Va": "variant",
    "Ve": "version",
    "Mk": "make",
    "Cn": "commercial_name",
    "m (kg)": "mass_in_running_order (kg)",
    "Ewltp (g/km)": "co2_emissions_WLTP (g/km)",
    "Ft": "Motor energy",
    "Fm": "fuel_mode",
    "ec (cm3)": "engine_capacity (cm3)",
    "ep (KW)": "engine_power (KW)",
    "z (Wh/km)": "electric_energy_consumption (Wh/km)",
    "r": "registrations",
    "year": "TIME_PERIOD"
}

for old_col, new_col in column_mapping.items():
    df = df.withColumnRenamed(old_col, new_col)

Drop non EU countries

In [ ]:
eu27_2020 = ['BE', 'BG', 'CZ', 'DK', 'DE', 'EE', 'IE', 'EL', 'ES', 'FR', 'HR', 'IT', 'CY', 'LV', 'LT', 'LU', 'HU', 'MT', 'NL', 'AT', 'PL', 'PT', 'RO', 'SI', 'SK', 'FI', 'SE']
country_dict = {code: coco.convert(names=code, to='name_short', not_found=None) for code in eu27_2020}

df = df.filter(df["geo"].isin(eu27_2020))

last_present = df.count()

In [ ]:
df.filter(F.col("co2_emissions_WLTP (g/km)").isNull() & F.col("Motor energy").contains("petrol") & ~F.col("version").isNull() & F.col("commercial_name").contains("CHALLENGER SRT")).show(10)

In [ ]:
df.filter(~F.col("co2_emissions_WLTP (g/km)").isNull() & F.col("commercial_name").contains("CHALLENGER SRT")).show(10)

Drop  N/A values

In [ ]:
num_cols = [
   "mass_in_running_order (kg)", "co2_emissions_WLTP (g/km)", 
    "engine_capacity (cm3)", "engine_power (KW)", "electric_energy_consumption (Wh/km)"
]
str_cols = ["geo", "commercial_name", "Motor energy"]
all_cols = str_cols + num_cols

df = df.na.drop(subset=str_cols)
df = df.filter(df["TIME_PERIOD"] != 0)

print(f"Row count changed by: {last_present - df.count()}")
last_present = df.count()
print(f"{last_present} rows present")

Drop columns we don't need

In [ ]:
not_needed = ["vehicle_family_id_number", "make", "fuel_mode", "type", "version"]
df = df.drop(*not_needed)

Clean the Motor Energy column to remove all various duplicates and unnecessary values, then group the various motor energy types in the ones we intend to analyse

In [ ]:
df = df.withColumn("Motor energy", 
    F.regexp_replace(F.trim(F.lower(F.col("Motor energy"))), "-", "/")
)
df.select("Motor energy").distinct().orderBy("Motor energy").rdd.map(lambda row: row[0]).collect()

In [ ]:
unnecessary = ["unknown", "other"]
df = df.filter(~df["Motor energy"].isin(unnecessary))

df = df.withColumn(
    "Motor energy",
    F.when(F.col("Motor energy") == "electric", "Electricity")
     .when(F.col("Motor energy").isin("petrol/electric", "hybrid/petrol/e", "petrol phev"), "Petrol hybrid")
     .when(F.col("Motor energy") == "diesel/electric", "Diesel hybrid")
     .when(F.col("Motor energy") == "petrol", "Petrol (excluding hybrids)")
     .when(F.col("Motor energy") == "diesel", "Diesel (excluding hybrids)")
     .otherwise("Alternative/Other") 
)

print(f"Row count changed by: {last_present - df.count()}")
last_present = df.count()
print(f"{last_present} rows present")

We try filling in some of the nulls by matching on similar vehicles based on commercial name, variant and motor energy, finding the closest match based on mass and engine capacity

In [ ]:
from pyspark.sql.window import Window

df_valid = df.filter(F.col("co2_emissions_WLTP (g/km)").isNotNull())
df_missing = df.filter(F.col("co2_emissions_WLTP (g/km)").isNull())

df_missing_evs = df_missing.filter(F.col("Motor energy") == "Electricity")
df_missing_non_evs = df_missing.filter(F.col("Motor energy") != "Electricity")

valid_refs = df_valid.filter(F.col("Motor energy") != "Electricity").select(
    F.col("commercial_name").alias("ref_commercial_name"),
    F.col("variant").alias("ref_variant"),
    F.col("Motor energy").alias("ref_Motor_energy"),
    F.col("mass_in_running_order (kg)").alias("ref_mass"),
    F.col("engine_capacity (cm3)").alias("ref_engine"),
    F.col("co2_emissions_WLTP (g/km)").alias("ref_co2")
).dropDuplicates()

df_missing_non_evs = df_missing_non_evs.withColumn("temp_row_id", F.monotonically_increasing_id())

joined_df = df_missing_non_evs.join(
    valid_refs,
    (df_missing_non_evs["commercial_name"] == valid_refs["ref_commercial_name"]) &
    (df_missing_non_evs["variant"] == valid_refs["ref_variant"]) &
    (df_missing_non_evs["Motor energy"] == valid_refs["ref_Motor_energy"]),
    "left"
)

joined_df = joined_df.withColumn(
    "distance",
    F.abs(F.col("mass_in_running_order (kg)") - F.col("ref_mass")) +
    F.abs(F.col("engine_capacity (cm3)") - F.col("ref_engine"))
)

window_spec = Window.partitionBy("temp_row_id").orderBy(F.col("distance").asc_nulls_last())

best_matches = joined_df.withColumn("rank", F.row_number().over(window_spec)) \
                        .filter(F.col("rank") == 1)

best_matches = best_matches.withColumn(
    "co2_emissions_WLTP (g/km)",
    F.coalesce(F.col("ref_co2"), F.col("co2_emissions_WLTP (g/km)"))
)

cols_to_drop = [
    "ref_commercial_name", "ref_variant", "ref_Motor_energy", 
    "ref_mass", "ref_engine", "ref_co2", "distance", "rank", "temp_row_id"
]
df_missing_imputed = best_matches.drop(*cols_to_drop)

df = df_valid.unionByName(df_missing_evs).unionByName(df_missing_imputed)

In [ ]:
nulls = df.filter(F.col("co2_emissions_WLTP (g/km)").isNull() & ~F.col("Motor energy").contains("Electricity"))
nulls.show(5), nulls.count()

As a fallback, we also try filling in some of the nulls by matching on similar vehicles based on commercial name and motor energy, finding the closest match based on mass, engine capacity and engine power

In [ ]:
df_valid = df.filter(F.col("co2_emissions_WLTP (g/km)").isNotNull())
df_missing = df.filter(F.col("co2_emissions_WLTP (g/km)").isNull())

df_missing_evs = df_missing.filter(F.col("Motor energy") == "Electricity")
df_missing_non_evs = df_missing.filter(F.col("Motor energy") != "Electricity")

valid_refs = df_valid.filter(F.col("Motor energy") != "Electricity").select(
    F.col("commercial_name").alias("ref_commercial_name"),
    F.col("Motor energy").alias("ref_Motor_energy"),
    F.col("mass_in_running_order (kg)").alias("ref_mass"),
    F.col("engine_capacity (cm3)").alias("ref_engine"),
    F.col("engine_power (KW)").alias("ref_power"),
    F.col("co2_emissions_WLTP (g/km)").alias("ref_co2")
).dropDuplicates()

df_missing_non_evs = df_missing_non_evs.withColumn("temp_row_id", F.monotonically_increasing_id())

joined_df = df_missing_non_evs.join(
    valid_refs,
    (df_missing_non_evs["commercial_name"] == valid_refs["ref_commercial_name"]) &
    (df_missing_non_evs["Motor energy"] == valid_refs["ref_Motor_energy"]),
    "left"
)

joined_df = joined_df.withColumn(
    "distance",
    F.abs(F.col("mass_in_running_order (kg)") - F.col("ref_mass")) +
    (F.abs(F.col("engine_capacity (cm3)") - F.col("ref_engine")) * 2) +
    (F.abs(F.col("engine_power (KW)") - F.col("ref_power")) * 5)
)

window_spec = Window.partitionBy("temp_row_id").orderBy(F.col("distance").asc_nulls_last())

best_matches = joined_df.withColumn("rank", F.row_number().over(window_spec)) \
                        .filter(F.col("rank") == 1)

best_matches = best_matches.withColumn(
    "co2_emissions_WLTP (g/km)",
    F.coalesce(F.col("ref_co2"), F.col("co2_emissions_WLTP (g/km)"))
)

cols_to_drop = [
    "ref_commercial_name", "ref_Motor_energy", "ref_mass", 
    "ref_engine", "ref_power", "ref_co2", "distance", "rank", "temp_row_id"
]
df_imputed_p2 = best_matches.drop(*cols_to_drop)

df = df_valid.unionByName(df_missing_evs).unionByName(df_imputed_p2)

In [ ]:
nulls = df.filter(F.col("co2_emissions_WLTP (g/km)").isNull() & ~F.col("Motor energy").contains("Electricity"))
nulls.show(5), nulls.count()

In [ ]:
df.filter(F.col("co2_emissions_WLTP (g/km)").isNull()).groupBy("Motor energy").count().orderBy(F.col("count").desc()).show()

Clean the manufacturer_name_eu_standard_denomination column, removing unnecessary values and reducing duplicates

In [ ]:
df = df.filter(~F.col("manufacturer_name_eu_standard_denomination").isin("DUPLICATE", "OUT OF SCOPE", "UNKNOWN", "duplicate", "unknown"))

name_map = {
    "AUDI HUNGARIA": "AUDI AG",
    "AUDI SPORT": "AUDI AG",
    "BEE": "BEE BEE",
    "BLUECAR ITALY": "BLUECAR",
    "BMW GMBH": "BMW AG",
    "DONGFENG LIUZHOU": "DONGFENG",
    "DONGFENG MOTOR": "DONGFENG",
    "DONKEVOORT": "DONKERVOORT",
    "DR MOTOR": "DR AUTOMOBILES",
    "Duplicate": "DUPLICATE",
    "FORD INDIA": "FORD MOTOR COMPANY",
    "FORD MOTOR AUSTRALIA": "FORD MOTOR COMPANY",
    "FORD WERKE GMBH": "FORD MOTOR COMPANY",
    "Ford Motor Company": "FORD MOTOR COMPANY",
    "GENERAL MOTORS COMPANY": "GENERAL MOTORS",
    "GENERAL MOTORS HOLDINGS": "GENERAL MOTORS",
    "GM ITALIA": "GENERAL MOTORS",
    "GM KOREA": "GENERAL MOTORS",
    "GM KOREA": "GENERAL MOTORS",
    "HONDA CHINA": "HONDA",
    "HONDA MOTOR CO": "HONDA",
    "HONDA THAILAND": "HONDA",
    "HONDA TURKIYE": "HONDA",
    "HONDA UK": "HONDA",
    "HYUNDAI ASSAN": "HYUNDAI",
    "HYUNDAI ASSAN": "HYUNDAI",
    "HYUNDAI CZECH": "HYUNDAI",
    "HYUNDAI CZECH": "HYUNDAI",
    "HYUNDAI EUROPE": "HYUNDAI",
    "HYUNDAI INDIA": "HYUNDAI",
    "JIANGXI JIANGLING": "JIANGLING MOTOR",
    "KIA SLOVAKIA": "KIA",
    "KIA SLOVAKIA": "KIA",
    "LADA FRANCE": "LADA",
    "LANZHOU ZHIDOU": "LANZHOU",
    "MAGYAR SUZUKI": "SUZUKI MOTOR CORPORATION",
    "MARUTI SUZUKI": "SUZUKI MOTOR CORPORATION",
    "MAZDA EUROPE": "MAZDA",
    "MERCEDES AMG": "MERCEDES-BENZ AG",
    "MERCEDES-AMG": "MERCEDES-BENZ AG",
    "NISSAN AUTOMOTIVE EUROPE": "NISSAN",
    "OPEL AUTOMOBILE": "OPEL",
    "QUATTRO": "AUDI AG",
    "RADICAL MOTOSPORT": "RADICAL MOTORSPORT",
    "ROLLS-ROYCE": "ROLLS ROYCE",
    "SAIC MAXUS": "SAIC MOTOR CORPORATION",
    "SAIC MOTOR": "SAIC MOTOR CORPORATION",
    "STELLANTIS EUROPE": "STELLANTIS AUTO",
    "SUZUKI THAILAND": "SUZUKI MOTOR CORPORATION",
    "TOYOTA MOTOR CORPORATION": "TOYOTA",
    "WUHAN LOTUS": "LOTUS"
}

df = df.replace(name_map, subset=["manufacturer_name_eu_standard_denomination"])
print(f"Row count changed by: {last_present - df.count()}")
last_present = df.count()
print(f"{last_present} rows present")

Aggregate registrations by merging rows that are now duplicated, construct a new standard geopoliticaly entity column like with the other datasets, then finally reorder columns and write the results to disk

In [ ]:
grouping_columns = [col for col in df.columns if col != "registrations"]
df = df.groupBy(*grouping_columns) \
               .agg(F.sum("registrations").alias("registrations"))

mapping_expr = F.create_map([F.lit(x) for x in chain(*country_dict.items())])
df = df.withColumn("Geopolitical entity (reporting)", mapping_expr[F.col("geo")])

df = df.withColumn("registrations", F.col("registrations").cast("integer"))

df = df.select(
    "geo",
    "Geopolitical entity (reporting)",
    "TIME_PERIOD",
    "registrations",
    "manufacturer_name_eu_standard_denomination",
    "commercial_name",
    "version",
    "Motor energy",
    "mass_in_running_order (kg)",
    "co2_emissions_WLTP (g/km)",
    "engine_capacity (cm3)",
    "engine_power (KW)",
    "electric_energy_consumption (Wh/km)"
)

print(f"Row count changed by: {last_present - df.count()}")
last_present = df.count()
print(f"{last_present} rows present")

# FROM HERE

> Probably zeros / null values need to be resolved before aggregation so we also have more columns to find the correct values to impute. 

> If not we can try to search for the same car 'commercial_name' and 'manufacturer_name_eu_standard_denomination' and other categories that enable to distinguish univoquely the car we can try to impute the missing values!

Restrict temporal coverage of both datasets to the years 2014-2023.

In [ ]:
df = df.filter((F.col("TIME_PERIOD") >= 2014) & (F.col("TIME_PERIOD") <= 2023))

Convert numeric columns to the correct data type.

> TODO: Check if 'int' is fine or we need to use 'double'!

In [ ]:
df = df.withColumn("mass_in_running_order (kg)", F.col("mass_in_running_order (kg)").cast("int")) \
       .withColumn("co2_emissions_WLTP (g/km)", F.col("co2_emissions_WLTP (g/km)").cast("int")) \
       .withColumn("engine_capacity (cm3)", F.col("engine_capacity (cm3)").cast("int")) \
       .withColumn("engine_power (KW)", F.col("engine_power (KW)").cast("int")) \
       .withColumn("electric_energy_consumption (Wh/km)", F.col("electric_energy_consumption (Wh/km)").cast("int"))

In [ ]:
df.show(10, truncate=False)

Check zeros / null values in the 'co2_emissions_WLTP (g/km)' column

> Most zeros are associated with 'Motor energy' = 'Electricity', some with 'Motor energy' = 'Alternative/Other'. If not needed for the dashboard we can decide wether to drop the 'Alternative/Other' category in the dataset.

> TODO: null values need to be imputed there are lots!

In [ ]:
df.filter(F.col("co2_emissions_WLTP (g/km)") == 0).groupBy("Motor energy").count().orderBy(F.col("count").desc()).show()

In [ ]:
df.filter(F.col("co2_emissions_WLTP (g/km)").isNull()).groupBy("Motor energy").count().orderBy(F.col("count").desc()).show()

In [ ]:
df.filter(F.col("co2_emissions_WLTP (g/km)").isNull()).groupBy("manufacturer_name_eu_standard_denomination").count().orderBy(F.col("count").desc()).show(truncate=False)

Check zeros / null values in the 'engine_capacity (cm3)' column

> No zeros found!

> TODO: null values, apart from the 'Motor energy' = 'Electricity' category,are also in some rows with 'Motor energy' in 'Petrol' and 'Diesel' categories!

In [ ]:
df.filter(F.col("engine_capacity (cm3)") == 0).groupBy("Motor energy").count().orderBy(F.col("count").desc()).show()

In [ ]:
df.filter(F.col("engine_capacity (cm3)").isNull()).groupBy("Motor energy").count().orderBy(F.col("count").desc()).show()

In [ ]:
df.filter(F.col("engine_capacity (cm3)").isNull()).filter(F.col("Motor energy") != "Electricity").show()

Check zeros / null values in the 'engine_power (KW)' column

> Few zeros associated with 'Motor energy' = 'Petrol (excluding hybrid)' and 'Motor energy' = 'Diesel (excluding hybrid)'. Are all from 'BMW AG'!

>

In [ ]:
df.filter(F.col("engine_power (KW)") == 0).groupBy("Motor energy").count().orderBy(F.col("count").desc()).show()

In [ ]:
df.filter(F.col("engine_power (KW)") == 0).show(30, truncate=False)

In [ ]:
df.filter(F.col("engine_power (KW)").isNull()).groupBy("Motor energy").count().orderBy(F.col("count").desc()).show()

Check zeros / null values in the 'electric_energy_consumption (Wh/km)' column

> TODO: categories that should not have values have them, and categories that should have values don't have them. Need to check if we can do something about this!

In [ ]:
df.filter(F.col("electric_energy_consumption (Wh/km)") == 0).groupBy("Motor energy").count().orderBy(F.col("count").desc()).show()

In [ ]:
df.filter(F.col("electric_energy_consumption (Wh/km)") != 0).groupBy("Motor energy").count().orderBy(F.col("count").desc()).show()

In [ ]:
df.filter(F.col("electric_energy_consumption (Wh/km)").isNull()).groupBy("Motor energy").count().orderBy(F.col("count").desc()).show()

Save the cleaned dataset to disk in parquet format.

In [ ]:
df.write.mode("overwrite") \
    .option("compression", "gzip") \
    .parquet(f"{CLEANED_DATA_DIR}/4c_eea_co2_emissions_from_passenger_cars-001.parquet")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pivoted_df = (
    df.groupBy("Motor energy")
      .pivot("TIME_PERIOD")
      .agg(F.round(F.sum("registrations") / 1000, 1))
      .fillna(0)
      .orderBy("Motor energy")
)

raw_rows = pivoted_df.collect()
columns = pivoted_df.columns
years = sorted([c for c in columns if c != "Motor energy"])

matrix_dict = {
    r["Motor energy"]: [r[y] for y in years]
    for r in raw_rows
}

heatmap_data = pd.DataFrame.from_dict(matrix_dict, orient="index", columns=years)

plt.figure(figsize=(12, 6))
ax = sns.heatmap(
    heatmap_data,
    annot=True,
    fmt="",
    annot_kws={"weight": "bold", "size": 9},
    cmap="YlOrRd",
    linewidths=0.5,
    linecolor="#2c3e50",
    cbar_kws={'label': 'Registrations (in Thousands)'}
)

# Heatmap border
for _, spine in ax.spines.items():
    spine.set_visible(True)
    spine.set_color("black")
    spine.set_linewidth(1.5)

plt.title("Total Car Registrations (in Thousands) by TIME_PERIOD and Motor energy [EU27_2020]", fontsize=14, pad=15)
plt.xlabel("TIME_PERIOD")
plt.ylabel("Motor energy category")
plt.xticks(rotation=0)
plt.yticks(rotation=0)

plt.tight_layout()
plt.savefig("img/powertrain_year_heatmap.png", dpi=300)
plt.show()